# Блокнот Colab: Веб-паук (краулер) для информационного поиска

# 1. Введение
# Веб-пауки (краулеры, роботы) — ключевой компонент поисковых систем. Они автоматически обходят веб-страницы,
# собирают их содержимое и передают его в индекс. В этом блокноте мы построим простого паука, который:
# - начинает с заданного URL,
# - извлекает все ссылки со страницы,
# - рекурсивно обходит их с учётом ограничений (глубина, количество страниц, robots.txt),
# - извлекает заголовок и основной текст,
# - сохраняет собранные данные для дальнейшего анализа.
#
# Внимание! Всегда соблюдайте правила вежливости: уважайте robots.txt, добавляйте задержки между запросами,
# не перегружайте серверы. В этом примере мы ограничим количество запросов.



In [ ]:
# ## 2. Установка и импорт библиотек

!pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import time
import pandas as pd
from collections import deque

In [ ]:
# 3. Класс-паук (краулер)

class WebCrawler:
    def __init__(self, start_url, max_pages=20, max_depth=5, delay=1):
        """
        Инициализация краулера.
        :param start_url: начальный URL
        :param max_pages: максимальное количество страниц для посещения
        :param max_depth: максимальная глубина обхода (0 - только стартовая)
        :param delay: задержка между запросами (секунды)
        """
        self.start_url = start_url
        self.max_pages = max_pages
        self.max_depth = max_depth
        self.delay = delay
        self.visited = set()
        # Очередь хранит (url, depth)
        self.queue = deque([(start_url, 0)])
        self.results = []
        self.robots_cache = {}

    def get_robot_parser(self, url):
        """Возвращает RobotFileParser для домена (кэшируется)."""
        parsed = urlparse(url)
        domain = f"{parsed.scheme}://{parsed.netloc}"
        if domain not in self.robots_cache:
            rp = RobotFileParser()
            rp.set_url(urljoin(domain, "/robots.txt"))
            try:
                rp.read()
            except:
                rp = None
            self.robots_cache[domain] = rp
        return self.robots_cache[domain]

    def can_fetch(self, url):
        """Проверяет robots.txt."""
        rp = self.get_robot_parser(url)
        if rp is None:
            return True
        return rp.can_fetch("MyCrawler", url)

    def fetch_page(self, url):
        """
        Загружает страницу по URL.
        Возвращает (html, status_code) или (None, status_code) при ошибке.
        """
        try:
            headers = {'User-Agent': 'MyCrawler (educational purpose)'}
            response = requests.get(url, headers=headers, timeout=5)
            status_code = response.status_code
            response.raise_for_status()
            content_type = response.headers.get('Content-Type', '')
            if 'text/html' not in content_type:
                print(f"  Пропускаем {url}: не HTML ({content_type})")
                return None, status_code
            return response.text, status_code
        except Exception as e:
            print(f"  Ошибка при загрузке {url}: {e}")
            # При любой ошибке возвращаем статус, если он был
            status_code = getattr(e, 'response', None)
            if isinstance(status_code, requests.Response):
                status_code = status_code.status_code
            else:
                status_code = 0
            return None, status_code

    def extract_links(self, html, base_url):
        """Извлекает ссылки из HTML, возвращает множество абсолютных URL."""
        soup = BeautifulSoup(html, 'html.parser')
        links = set()
        for a in soup.find_all('a', href=True):
            href = a['href']
            if not href or href.startswith('#') or href.startswith('javascript:'):
                continue
            absolute = urljoin(base_url, href)
            if absolute.startswith(('http://', 'https://')):
                links.add(absolute)
        return links

    def extract_info(self, html, url, status_code, depth):
        """Извлекает заголовок и текст, добавляет статус и глубину."""
        soup = BeautifulSoup(html, 'html.parser')
        title = soup.find('title')
        title_text = title.get_text(strip=True) if title else "Без заголовка"

        for script in soup(["script", "style"]):
            script.decompose()
        text = soup.get_text(separator=' ', strip=True)
        text_preview = text[:500] + "..." if len(text) > 500 else text

        return {
            'url': url,
            'title': title_text,
            'text_preview': text_preview,
            'length': len(text),
            'status_code': status_code,
            'depth': depth
        }

    def crawl(self):
        """Запускает процесс обхода."""
        print(f"Начинаем обход с {self.start_url}")
        print(f"Максимум страниц: {self.max_pages}, глубина: {self.max_depth}, задержка: {self.delay} сек.")

        while self.queue and len(self.visited) < self.max_pages:
            url, depth = self.queue.popleft()
            if url in self.visited:
                continue

            # Проверка robots.txt
            if not self.can_fetch(url):
                print(f"Пропускаем {url} (запрещено robots.txt)")
                self.visited.add(url)
                continue

            print(f"Загружаем {url} (глубина {depth})...")
            html, status_code = self.fetch_page(url)
            if html is None:
                # Даже если не удалось загрузить, записываем запись с ошибкой
                self.results.append({
                    'url': url,
                    'title': "Ошибка загрузки",
                    'text_preview': "",
                    'length': 0,
                    'status_code': status_code,
                    'depth': depth
                })
                self.visited.add(url)
                continue

            # Успешно загрузили – извлекаем информацию
            info = self.extract_info(html, url, status_code, depth)
            self.results.append(info)
            self.visited.add(url)

            # Если достигли максимальной глубины, не добавляем новые ссылки
            if depth >= self.max_depth:
                continue

            # Извлекаем ссылки и добавляем в очередь с глубиной +1
            new_links = self.extract_links(html, url)
            for link in new_links:
                if link not in self.visited:
                    self.queue.append((link, depth + 1))

            print(f"  Посещено: {len(self.visited)} страниц, в очереди: {len(self.queue)}")
            time.sleep(self.delay)

        print(f"Обход завершён. Всего обработано страниц: {len(self.results)}")
        return self.results

In [ ]:
# 4. Запуск краулера на примере Википедии
# Выберем начальную страницу — статью об информационном поиске.
# Википедия разрешает краулинг в robots.txt (с задержкой 1 сек).
# Установим небольшую глубину обхода.

#start_url = "https://ru.wikipedia.org/wiki/Информационный_поиск"
start_url = "http://www.cs.vsu.ru"
crawler = WebCrawler(start_url, max_pages=30, delay=1)
data = crawler.crawl()

Начинаем обход с http://www.cs.vsu.ru
Максимум страниц: 30, глубина: 5, задержка: 1 сек.
Загружаем http://www.cs.vsu.ru (глубина 0)...
  Посещено: 1 страниц, в очереди: 90
Загружаем https://www.cs.vsu.ru/for-entrants/ (глубина 1)...
  Посещено: 2 страниц, в очереди: 163
Загружаем https://www.cs.vsu.ru/2025/06/chempionat-po-programmirovaniyu-i-kompyuternoj-bezopasnosti-cybercraft-bitva-kodov/ (глубина 1)...
  Посещено: 3 страниц, в очереди: 255
Загружаем https://www.cs.vsu.ru/for-students/student-research-books/ (глубина 1)...
  Посещено: 4 страниц, в очереди: 337
Пропускаем https://www.linkedin.com/groups/4735012 (запрещено robots.txt)
Загружаем https://www.cs.vsu.ru/2013/09/posvyashhenie-v-studenty-2013/ (глубина 1)...
  Посещено: 6 страниц, в очереди: 418
Загружаем http://vk.com/cs_vsu (глубина 1)...
  Посещено: 7 страниц, в очереди: 418
Загружаем https://www.cs.vsu.ru/for-students/elektronnye-resursy-vgu/ (глубина 1)...
  Посещено: 8 страниц, в очереди: 487
Загружаем https://www.cs.

/tmp/ipykernel_7015/2841005075.py:83: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, 'html.parser')
/tmp/ipykernel_7015/2841005075.py:70: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might

  Посещено: 19 страниц, в очереди: 1076
Загружаем https://www.cs.vsu.ru/for-entrants/faq-for-entrants/ (глубина 1)...
  Посещено: 20 страниц, в очереди: 1139
Загружаем https://www.cs.vsu.ru/itmarathon/project-definition/ (глубина 1)...
  Посещено: 21 страниц, в очереди: 1198
Загружаем https://www.cs.vsu.ru/2021/06/iii-otkrytye-mezhvuzovskie-sorevnovaniya-po-programmirovaniyu-i-kompyuternoj-bezopasnosti-vrnctf/ (глубина 1)...
  Посещено: 22 страниц, в очереди: 1273
Загружаем https://www.cs.vsu.ru/2025/04/den-otkrytyx-dverej-3/ (глубина 1)...
  Посещено: 23 страниц, в очереди: 1332
Загружаем http://www.ddt.cs.vsu.ru (глубина 1)...
  Посещено: 24 страниц, в очереди: 1386
Загружаем https://www.cs.vsu.ru/2025/08/o-nachale-uchebnogo-goda-4/ (глубина 1)...
  Посещено: 25 страниц, в очереди: 1443
Загружаем https://www.youtube.com/playlist?list=PLJjoStu1QQcMFIHPxYcBZMPGpcS4tedoB (глубина 1)...
  Посещено: 26 страниц, в очереди: 1456
Загружаем https://www.cs.vsu.ru/mprog2022/ (глубина 1)...
  По

In [ ]:
# ## 5. Анализ собранных данных
# Преобразуем список словарей в DataFrame для удобного просмотра.

df = pd.DataFrame(data)
print(f"Собрано {len(df)} страниц")
df.head()

Собрано 28 страниц


,url,title,text_preview,length,status_code,depth
0,http://www.cs.vsu.ru,Сайт Факультета компьютерных наук ВГУ,Сайт Факультета компьютерных наук ВГУ Расписан...,7806,200,0
1,https://www.cs.vsu.ru/for-entrants/,Абитуриентам | Факультет компьютерных наук,Абитуриентам | Факультет компьютерных наук Рас...,4443,200,1
2,https://www.cs.vsu.ru/2025/06/chempionat-po-pr...,Чемпионат по программированию и компьютерной б...,Чемпионат по программированию и компьютерной б...,12486,200,1
3,https://www.cs.vsu.ru/for-students/student-res...,Научные труды студентов | Факультет компьютерн...,Научные труды студентов | Факультет компьютерн...,4820,200,1
4,https://www.cs.vsu.ru/2013/09/posvyashhenie-v-...,Посвящение в студенты 2013 | Факультет компьют...,Посвящение в студенты 2013 | Факультет компьют...,4378,200,1


In [ ]:
# ## 6. Простое ранжирование по длине текста
# В реальных поисковых системах ранжирование сложнее, но для демонстрации отсортируем страницы по длине текста.

df_sorted = df.sort_values(by='length', ascending=False)
df_sorted[['title', 'url', 'length']].head()

,title,url,length
16,Факультет компьютерных наук,http://www.cs.vsu.ru/feed/,107476
25,Результаты секции НОУ «КНиИТ» на ФКН | Факульт...,https://www.cs.vsu.ru/2025/04/rezultaty-sekcii...,20304
26,Итоги работы секции НОУ ФКН «Компьютерные наук...,https://www.cs.vsu.ru/2021/05/nou-fkn-2021/,19254
2,Чемпионат по программированию и компьютерной б...,https://www.cs.vsu.ru/2025/06/chempionat-po-pr...,12486
19,III Открытые межвузовские соревнования по прог...,https://www.cs.vsu.ru/2021/06/iii-otkrytye-mez...,11073


In [ ]:
# ## 7. Сохранение результатов
# Можно сохранить собранные данные в файл CSV для дальнейшего использования.

df.to_csv('crawled_pages.csv', index=False, encoding='utf-8', sep=';')
print("Данные сохранены в crawled_pages.csv")

Данные сохранены в crawled_pages.csv
